من کد این روش مدلسازی را با دو روش بدون تابع بازگشتی و با تابع زدم

In [ ]:
# نصب پکیج‌های مورد نیاز
!pip install cplex
!pip install docplex

# وارد کردن کتابخانه‌ها
from docplex.mp.model import Model

# مرحله ۱: تعریف مدل
model = Model(name='ALM_Stochastic')

# مرحله ۲: تعریف مجموعه‌ها
I = ['Stock', 'Bond']  # دارایی‌ها
T = range(4)           # سال‌ها (0 تا 3)
S = range(8)           # سناریوها (0 تا 7)

# مرحله ۳: تعریف پارامترها
w0 = 55000             # سرمایه اولیه
L = 80000              # بدهی نهایی
prob = {s: 0.125 for s in S}  # احتمال هر سناریو
q = 1.0                # وزن سود
r = -1.0               # وزن زیان

# مرحله ۴: تعریف بازده‌ها
R = {}
for s in S:
    for i in I:
        for t in range(1, 4):
            if t == 1:
                state = 'Good' if s in [0, 1, 2, 3] else 'Bad'
            elif t == 2:
                state = 'Good' if s in [0, 1, 4, 5] else 'Bad'
            else:
                state = 'Good' if s in [0, 2, 4, 6] else 'Bad'
            R[(i, t, s)] = 1.25 if (i == 'Stock' and state == 'Good') else \
                          1.06 if (i == 'Stock' and state == 'Bad') else \
                          1.14 if (i == 'Bond' and state == 'Good') else 1.12

# مرحله ۵: تعریف متغیرها
x = {(i, t, s): model.continuous_var(lb=0, name=f'x_{i}_{t}_{s}') for i in I for t in T for s in S}
w_plus = {s: model.continuous_var(lb=0, name=f'w_plus_{s}') for s in S}
w_minus = {s: model.continuous_var(lb=0, name=f'w_minus_{s}') for s in S}

# مرحله ۶: تعریف تابع هدف
objective = model.sum(prob[s] * (q * w_plus[s] + r * w_minus[s]) for s in S)
model.maximize(objective)

# مرحله ۷: تعریف محدودیت‌ها
# 1. بودجه اولیه
for s in S:
    model.add_constraint(model.sum(x[i, 0, s] for i in I) == w0, ctname=f'initial_budget_{s}')

# 2. تعادل سرمایه
for t in T:
    if t == 0:
        continue
    for s in S:
        model.add_constraint(
            model.sum(x[i, t, s] for i in I) == model.sum(R[i, t, s] * x[i, t-1, s] for i in I),
            ctname=f'balance_{t}_{s}'
        )

# 3. بدهی نهایی
for s in S:
    model.add_constraint(
        model.sum(R[i, 3, s] * x[i, 2, s] for i in I) == L + w_plus[s] - w_minus[s],
        ctname=f'liability_{s}'
    )

# 4. شرط عدم پیش‌بینی
for i in I:
    for s1 in [0, 1, 2, 3]:
        for s2 in [0, 1, 2, 3]:
            if s1 < s2:
                model.add_constraint(x[i, 1, s1] == x[i, 1, s2], ctname=f'na_t1_{i}_{s1}_{s2}')
    for s1 in [4, 5, 6, 7]:
        for s2 in [4, 5, 6, 7]:
            if s1 < s2:
                model.add_constraint(x[i, 1, s1] == x[i, 1, s2], ctname=f'na_t1_{i}_{s1}_{s2}')

# سال ۲: مثلاً 0-1 (GG), 2-3 (GB), 4-5 (BG), 6-7 (BB)
for i in I:
    for (s1, s2) in [(0, 1), (2, 3), (4, 5), (6, 7)]:
        model.add_constraint(x[i, 2, s1] == x[i, 2, s2], ctname=f'na_t2_{i}_{s1}_{s2}')

# مرحله ۸: حل مدل با CPLEX
solution = model.solve()
 # بررسی و نمایش نتایج
if solution:
    for i in I:
        for t in T:
            for s in S:
                print(f"x[{i}, {t}, {s}] = {x[i, t, s].solution_value}")
    print("\nسود (w_plus):")
    for s in S:
        print(f"w_plus[{s}] = {w_plus[s].solution_value}")
    print("\nزیان (w_minus):")
    for s in S:
        print(f"w_minus[{s}] = {w_minus[s].solution_value}")

    # نمایش مقدار تابع هدف
    print("\nمقدار تابع هدف:")
    print(f"Objective Value = {model.objective_value}")
else:
    print("حل بهینه پیدا نشد.")


x[Stock, 0, 0] = 55000.0
x[Stock, 0, 1] = 55000.0
x[Stock, 0, 2] = 55000.0
x[Stock, 0, 3] = 55000.0
x[Stock, 0, 4] = 0
x[Stock, 0, 5] = 0
x[Stock, 0, 6] = 0
x[Stock, 0, 7] = 0
x[Stock, 1, 0] = 68750.0
x[Stock, 1, 1] = 68750.0
x[Stock, 1, 2] = 68750.0
x[Stock, 1, 3] = 68750.0
x[Stock, 1, 4] = 61600.00000000001
x[Stock, 1, 5] = 61600.00000000001
x[Stock, 1, 6] = 61600.00000000001
x[Stock, 1, 7] = 61600.00000000001
x[Stock, 2, 0] = 85937.5
x[Stock, 2, 1] = 85937.5
x[Stock, 2, 2] = 72875.0
x[Stock, 2, 3] = 72875.0
x[Stock, 2, 4] = 77000.00000000001
x[Stock, 2, 5] = 77000.00000000001
x[Stock, 2, 6] = 65296.000000000015
x[Stock, 2, 7] = 65296.000000000015
x[Stock, 3, 0] = 0
x[Stock, 3, 1] = 0
x[Stock, 3, 2] = 0
x[Stock, 3, 3] = 0
x[Stock, 3, 4] = 0
x[Stock, 3, 5] = 0
x[Stock, 3, 6] = 0
x[Stock, 3, 7] = 0
x[Bond, 0, 0] = 0
x[Bond, 0, 1] = 0
x[Bond, 0, 2] = 0
x[Bond, 0, 3] = 0
x[Bond, 0, 4] = 55000.0
x[Bond, 0, 5] = 55000.0
x[Bond, 0, 6] = 55000.0
x[Bond, 0, 7] = 55000.0
x[Bond, 1, 0] = 0
x[Bo

In [ ]:
#%%capture
# نصب کتابخانه‌های مورد نیاز
!pip install pyomo
!apt-get install -y -qq glpk-utils

# وارد کردن کتابخانه‌ها
from pyomo.environ import *
import numpy as np
import pandas as pd

########۳ مرحله ۱: تعریف مدل

model = ConcreteModel(name='ALM_Stochastic_Recursive')

##########################

# مرحله ۲: تعریف مجموعه‌ها

model.I = Set(initialize=['Stock', 'Bond'])  # دارایی‌ها
model.T = RangeSet(0, 3)  # تعریف مجموعه سال‌ها از 0 تا 3
model.S = RangeSet(0, 7)  # سناریوها (0 تا 7)
scenario_names = ['GGG', 'GGB', 'GBG', 'GBB', 'BGG', 'BGB', 'BBG', 'BBB']
print("T:", list(model.T))
print("S:", list(model.S))

############################

# مرحله ۳: تعریف پارامترها
model.w0 = Param(initialize=55000)           # سرمایه اولیه
model.L = Param(initialize=80000)            # بدهی نهایی
model.prob = Param(model.S, initialize={s: 0.125 for s in range(8)})  # احتمال هر سناریو
model.q = Param(initialize=1)              # وزن سود
model.r = Param(initialize=1)             # وزن زیان

print("\nپارامترها:")
print("w0:", value(model.w0))
print("L:", value(model.L))
print("prob:", {s: value(model.prob[s]) for s in model.S})
print("q:", value(model.q))
print("r:", value(model.r))

############################

# تعریف بازده‌ها
def calculate_return(i, t, s):
    if t == 1:
        state = 'Good' if s in range(4) else 'Bad'
    elif t == 2:
        state = 'Good' if s in [0, 1, 4, 5] else 'Bad'
    else:
        state = 'Good' if s in [0, 2, 4, 6] else 'Bad'

    if i == 'Stock':
        return 1.25 if state == 'Good' else 1.06
    else:
        return 1.14 if state == 'Good' else 1.12

model.R = Param(model.I, RangeSet(1, 3), model.S, initialize=lambda model, i, t, s: calculate_return(i, t, s))

############################

# مرحله ۴: تعریف متغیرها
model.x = Var(model.I, model.T, model.S, domain=NonNegativeReals)  # سرمایه‌گذاری
model.W = Var(model.T, model.S, domain=NonNegativeReals)           # ثروت
model.w_plus = Var(model.S, domain=NonNegativeReals)               # سود
model.w_minus = Var(model.S, domain=NonNegativeReals)              # زیان

############################

# مرحله ۵: تعریف تابع هدف
def objective_rule(m):
    return quicksum(m.prob[s] * (m.q * m.w_plus[s] - m.r * m.w_minus[s]) for s in m.S)
model.obj = Objective(rule=objective_rule, sense=maximize)
print("تابع هدف تعریف شد: ماکزیمم کردن سود وزن‌دار.")

############################

# مرحله ۶: تعریف محدودیت‌ها

# ثروت اولیه
def initial_wealth_rule(m, s):
    return m.W[0, s] == m.w0
model.initial_wealth = Constraint(model.S, rule=initial_wealth_rule)

############################

# تعادل ثروت در هر سال
def wealth_balance_rule(m, t, s):
    if t == 0:
        return Constraint.Skip
    return m.W[t, s] == sum(m.R[i, t, s] * m.x[i, t-1, s] for i in m.I)
model.wealth_balance = Constraint(model.T, model.S, rule=wealth_balance_rule)

############################

# تخصیص ثروت
def allocation_rule(m, t, s):
    return sum(m.x[i, t, s] for i in m.I) == m.W[t, s]
model.allocation = Constraint(model.T, model.S, rule=allocation_rule)

############################

# بدهی نهایی
def liability_rule(m, s):
    return m.W[3, s] == m.L + m.w_plus[s] - m.w_minus[s]
model.liability = Constraint(model.S, rule=liability_rule)

############################

# شرط عدم پیش‌بینی (Non-Anticipativity)
def non_anticipativity_t1_rule(m, i, s1, s2):
    if s1 < s2 and (s1 in range(4) and s2 in range(4) or s1 in range(4, 8) and s2 in range(4, 8)):
        return m.x[i, 1, s1] == m.x[i, 1, s2]
    return Constraint.Skip
model.na_t1 = Constraint(model.I, model.S, model.S, rule=non_anticipativity_t1_rule)

############################

def non_anticipativity_t2_rule(m, i, s1, s2):
    if (s1, s2) in [(0, 1), (2, 3), (4, 5), (6, 7)]:
        return m.x[i, 2, s1] == m.x[i, 2, s2]
    return Constraint.Skip
model.na_t2 = Constraint(model.I, model.S, model.S, rule=non_anticipativity_t2_rule)

############################

# مرحله ۷: حل مدل با GLPK
solver = SolverFactory('glpk', executable='/usr/bin/glpsol')
results = solver.solve(model)
############################
# حل مدل با GLPK
solver = SolverFactory('glpk', executable='/usr/bin/glpsol')
results = solver.solve(model)

# بررسی وضعیت حل
print("وضعیت حل:", results.solver.status)
print("وضعیت خاتمه:", results.solver.termination_condition)

# استخراج مقادیر متغیرها از مدل
W_values = {}
for t in model.T:
    for s in model.S:
        W_values[(t, s)] = value(model.W[t, s])

x_values = {}
for i in model.I:
    for t in model.T:
        for s in model.S:
            x_values[(i, t, s)] = value(model.x[i, t, s])

# تعریف تابع بازگشتی (Value Function)
def get_next_scenarios(t, s):
    if t == 2:
        return [0, 1, 4, 5] if s in [0, 1, 4, 5] else [2, 3, 6, 7]
    elif t == 1:
        return [0, 1, 2, 3] if s in [0, 1, 4, 5] else [4, 5, 6, 7]
    else:
        return [s]

V = {}
for t in range(3, -1, -1):
    for s in model.S:
        W_t = W_values[(t, s)]
        if t == 3:
            w_plus = model.w_plus[s].value if model.w_plus[s].value is not None else 0
            w_minus = model.w_minus[s].value if model.w_minus[s].value is not None else 0
            V[(t, W_t, s)] = q * w_plus - r * w_minus
        else:
            next_scenarios = get_next_scenarios(t, s)
            value = sum(prob * V.get((t+1, W_values[(t+1, s_next)], s_next), 0) for s_next in next_scenarios)
            V[(t, W_t, s)] = value

# نمایش نتایج
print("\nتابع ارزش در هر مرحله:")
V_data = {}
for t in T:
    for s in S:
        W_t = W_values[(t, s)]
        V_data[f"V[{t}, {W_t:.2f}, {scenario_names[s]}]"] = V.get((t, W_t, s), 0)
pprint(V_data)

print("\nتصمیم‌های بهینه (x):")
x_data = []
for i in model.I:
    for t in model.T:
        for s in model.S:
            x_data.append({
                'Asset': i,
                'Year': t,
                'Scenario': scenario_names[s],
                'Investment': model.x[i, t, s].value if model.x[i, t, s].value is not None else 0
            })
x_df = pd.DataFrame(x_data)
pprint(x_df)

print("\nجدول سود و زیان:")
w_data = []
for s in model.S:
    w_data.append({
        'Scenario': scenario_names[s],
        'Surplus (w_plus)': model.w_plus[s].value if model.w_plus[s].value is not None else 0,
        'Shortfall (w_minus)': model.w_minus[s].value if model.w_minus[s].value is not None else 0
    })
w_df = pd.DataFrame(w_data)
pprint(w_df)

print("\nمقدار تابع هدف:")
try:
    from pyomo.environ import value
    obj_value = value(model.obj)
    print(f"مقدار تابع هدف: {obj_value}")
except Exception as e:
    print(f"خطا در محاسبه مقدار تابع هدف: {e}")



T: [0, 1, 2, 3]
S: [0, 1, 2, 3, 4, 5, 6, 7]

پارامترها:
w0: 55000
L: 80000
prob: {0: 0.125, 1: 0.125, 2: 0.125, 3: 0.125, 4: 0.125, 5: 0.125, 6: 0.125, 7: 0.125}
q: 1
r: 1
تابع هدف تعریف شد: ماکزیمم کردن سود وزن‌دار.
وضعیت حل: ok
وضعیت خاتمه: optimal

تابع ارزش در هر مرحله:
{'V[0, 55000.00, BBB]': 322.80513671875076,
 'V[0, 55000.00, BBG]': 322.80513671875076,
 'V[0, 55000.00, BGB]': 322.80513671875076,
 'V[0, 55000.00, BGG]': 322.80513671875076,
 'V[0, 55000.00, GBB]': 322.80513671875076,
 'V[0, 55000.00, GBG]': 322.80513671875076,
 'V[0, 55000.00, GGB]': 322.80513671875076,
 'V[0, 55000.00, GGG]': 322.80513671875076,
 'V[1, 61600.00, BBB]': 2582.441093750006,
 'V[1, 61600.00, BBG]': 2582.441093750006,
 'V[1, 61600.00, BGB]': 2582.441093750006,
 'V[1, 61600.00, BGG]': 2582.441093750006,
 'V[1, 68750.00, GBB]': 2582.441093750006,
 'V[1, 68750.00, GBG]': 2582.441093750006,
 'V[1, 68750.00, GGB]': 2582.441093750006,
 'V[1, 68750.00, GGG]': 2582.441093750006,
 'V[2, 65296.00, BBB]': 3281.